# Phase 3 — Train Models (PPR)
Trains Ridge Regression and XGBoost for each position (QB, RB, WR, TE).
Uses walk-forward cross-validation to evaluate accuracy on 2024 and 2025.
Final models are trained on all seasons and used to predict 2026 PPG.

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

INPUT_DIR  = '../PickleFiles/NewModel'
OUTPUT_DIR = '../PickleFiles/NewModel'

# Columns to never use as features
EXCLUDE_COLS = {
    'player_name', 'team', 'position', 'age_bucket',
    'season', 'next_ppg', 'ppg', 'fantasy_pts',
    'ppg_lag1', 'ppg_lag2'  # intermediate cols from weighted_ppg calc
}

# Walk-forward test seasons (we know ground truth for these)
TEST_SEASONS = [2024, 2025]

print('Imports done.')

Imports done.


In [2]:
# ── Helper Functions ──────────────────────────────────────────────────────────

def create_target(df, id_col='player_name', season_col='season', ppg_col='ppg'):
    """
    Add next_ppg = the player's PPG in their next season.
    Rows where next_ppg is NaN = the player had no following season in the dataset.
    The 2025 feature rows will have next_ppg=NaN — these are what we predict for 2026.
    """
    df = df.sort_values([id_col, season_col]).copy()
    df['next_ppg'] = df.groupby(id_col)[ppg_col].shift(-1)
    return df


def get_feature_cols(df):
    """Return all numeric columns not in EXCLUDE_COLS."""
    numeric = df.select_dtypes(include=[np.number]).columns.tolist()
    return [c for c in numeric if c not in EXCLUDE_COLS]


def make_ridge():
    """Ridge pipeline: median impute NaN, standardize, then Ridge."""
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   Ridge(alpha=10.0))
    ])


def make_xgb():
    """XGBoost — handles NaN natively, capped depth to avoid overfitting on small datasets."""
    return xgb.XGBRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbosity=0
    )


def walk_forward_cv(df, feature_cols, test_seasons):
    """
    Walk-forward cross-validation.
    For each test_season:
      - Train on all feature_seasons < (test_season - 1), target = next season ppg
      - Test on feature_season = (test_season - 1), target = test_season ppg
    Returns a dict of {test_season: {'ridge_mae': x, 'xgb_mae': y, 'blend_mae': z, 'n': n}}
    """
    results = {}

    for test_season in test_seasons:
        feature_season = test_season - 1  # the season whose features predict test_season

        train = df[(df['season'] < feature_season) & df['next_ppg'].notna()]
        test  = df[(df['season'] == feature_season) & df['next_ppg'].notna()]

        if len(train) < 10 or len(test) < 3:
            print(f'  Skipping {test_season}: insufficient data (train={len(train)}, test={len(test)})')
            continue

        X_train = train[feature_cols]
        y_train = train['next_ppg']
        X_test  = test[feature_cols]
        y_test  = test['next_ppg']

        # Ridge
        ridge = make_ridge()
        ridge.fit(X_train, y_train)
        ridge_preds = ridge.predict(X_test)
        ridge_mae   = mean_absolute_error(y_test, ridge_preds)

        # XGBoost
        xgb_model = make_xgb()
        xgb_model.fit(X_train, y_train)
        xgb_preds = xgb_model.predict(X_test)
        xgb_mae   = mean_absolute_error(y_test, xgb_preds)

        # Blend (50/50)
        blend_preds = (ridge_preds + xgb_preds) / 2
        blend_mae   = mean_absolute_error(y_test, blend_preds)

        results[test_season] = {
            'ridge_mae': round(ridge_mae, 2),
            'xgb_mae':   round(xgb_mae, 2),
            'blend_mae': round(blend_mae, 2),
            'n':         len(test)
        }

    return results


def pick_strategy(cv_results):
    """
    Compare average MAE across CV folds.
    If difference between winner and blend is < 5%, use blend (safer).
    Otherwise use the clear winner.
    """
    if not cv_results:
        return 'blend'

    avg = {'ridge': 0, 'xgb': 0, 'blend': 0}
    for r in cv_results.values():
        avg['ridge'] += r['ridge_mae']
        avg['xgb']   += r['xgb_mae']
        avg['blend']  += r['blend_mae']

    n = len(cv_results)
    avg = {k: v / n for k, v in avg.items()}
    best = min(avg, key=avg.get)
    second = sorted(avg, key=avg.get)[1]

    # If best is within 5% of blend, prefer blend for stability
    if best != 'blend' and abs(avg[best] - avg['blend']) / avg['blend'] < 0.05:
        return 'blend'
    return best


def train_final_and_predict(df, feature_cols, strategy, predict_season=2025):
    """
    Train final model on ALL seasons with known next_ppg.
    Predict next year PPG for players in predict_season.
    Returns (predictions_df, ridge_model, xgb_model)
    """
    train = df[df['next_ppg'].notna()]
    pred  = df[df['season'] == predict_season].copy()

    X_train = train[feature_cols]
    y_train = train['next_ppg']
    X_pred  = pred[feature_cols]

    ridge = make_ridge()
    ridge.fit(X_train, y_train)

    xgb_model = make_xgb()
    xgb_model.fit(X_train, y_train)

    ridge_preds = ridge.predict(X_pred)
    xgb_preds   = xgb_model.predict(X_pred)
    blend_preds = (ridge_preds + xgb_preds) / 2

    strategy_preds = {'ridge': ridge_preds, 'xgb': xgb_preds, 'blend': blend_preds}[strategy]

    pred = pred[['player_name', 'team', 'season', 'age', 'weighted_ppg', 'ppg', 'games']].copy()
    pred['predicted_ppg_2026'] = np.maximum(strategy_preds, 0)  # clip negatives
    pred['ridge_pred']         = np.maximum(ridge_preds, 0)
    pred['xgb_pred']           = np.maximum(xgb_preds, 0)
    pred['strategy']           = strategy
    pred = pred.sort_values('predicted_ppg_2026', ascending=False).reset_index(drop=True)

    return pred, ridge, xgb_model


def run_position(name, full_pkl, feature_pkl):
    """Full pipeline for one position: CV -> pick strategy -> final model -> predictions."""
    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')

    df      = pd.read_pickle(f'{INPUT_DIR}/{full_pkl}')
    df      = create_target(df)
    feat_cols = get_feature_cols(df)
    print(f'  Player-seasons: {len(df)} | Features: {len(feat_cols)}')
    print(f'  Training rows (next_ppg known): {df["next_ppg"].notna().sum()}')

    # Walk-forward CV
    print('\n  Walk-forward CV:')
    cv = walk_forward_cv(df, feat_cols, TEST_SEASONS)
    for season, r in cv.items():
        print(f'    Predicting {season} (n={r["n"]}): '
              f'Ridge MAE={r["ridge_mae"]:.2f} | '
              f'XGB MAE={r["xgb_mae"]:.2f} | '
              f'Blend MAE={r["blend_mae"]:.2f}')

    strategy = pick_strategy(cv)
    print(f'\n  Selected strategy: {strategy.upper()}')

    # Final model + 2026 predictions
    preds, ridge_model, xgb_model = train_final_and_predict(df, feat_cols, strategy)

    print(f'\n  2026 Predictions (top 10):')
    print(preds[['player_name','team','age','predicted_ppg_2026','weighted_ppg']]
          .head(10).to_string(index=False))

    # Save
    pos = name.lower()
    preds.to_pickle(f'{OUTPUT_DIR}/{pos}_predictions_ppr.pkl')
    with open(f'{OUTPUT_DIR}/{pos}_ridge_ppr.pkl', 'wb') as f:
        pickle.dump(ridge_model, f)
    with open(f'{OUTPUT_DIR}/{pos}_xgb_ppr.pkl', 'wb') as f:
        pickle.dump(xgb_model, f)

    return preds, cv


print('Helper functions defined.')

Helper functions defined.


In [3]:
# ── QB ────────────────────────────────────────────────────────────────────────
qb_preds, qb_cv = run_position('QB', 'qb_full.pkl', 'qb_features.pkl')


  QB
  Player-seasons: 126 | Features: 116
  Training rows (next_ppg known): 61

  Walk-forward CV:
    Predicting 2024 (n=12): Ridge MAE=2.27 | XGB MAE=2.82 | Blend MAE=2.54
    Predicting 2025 (n=5): Ridge MAE=4.36 | XGB MAE=4.21 | Blend MAE=4.28

  Selected strategy: BLEND

  2026 Predictions (top 10):
    player_name team  age  predicted_ppg_2026  weighted_ppg
 Baker Mayfield   TB 30.0           17.749373     14.778799
Patrick Mahomes   KC 29.0           17.399044     20.120000
  Justin Fields  NYJ 26.0           17.336865     16.358318
  Lamar Jackson  BAL 28.0           17.001671     17.711599
   Dak Prescott  DAL 32.0           16.924343     17.412368
    Jaxson Dart  NYG 22.0           16.800137     20.298333
   Daniel Jones  IND 28.0           15.888588     16.878069
Shedeur Sanders  CLE 23.0           15.549588     10.862500
 Marcus Mariota  WAS 31.0           15.447507     13.561538
   Tyler Shough   NO 25.0           14.708152     15.996000


In [4]:
# ── RB ────────────────────────────────────────────────────────────────────────
rb_preds, rb_cv = run_position('RB', 'rb_full.pkl', 'rb_features.pkl')


  RB
  Player-seasons: 264 | Features: 103
  Training rows (next_ppg known): 114

  Walk-forward CV:
    Predicting 2024 (n=29): Ridge MAE=3.50 | XGB MAE=3.63 | Blend MAE=3.48
    Predicting 2025 (n=14): Ridge MAE=3.58 | XGB MAE=3.76 | Blend MAE=3.66

  Selected strategy: BLEND

  2026 Predictions (top 10):
     player_name team  age  predicted_ppg_2026  weighted_ppg
    Jahmyr Gibbs  DET 23.0           18.027151     21.682353
   Ashton Jeanty   LV 21.0           17.858375     14.535294
   De'Von Achane  MIA 23.0           15.325423     19.475967
    Bucky Irving   TB 23.0           15.125631     14.050000
  Bijan Robinson  ATL 23.0           14.919267     21.170588
 Jonathan Taylor  IND 26.0           14.631356     20.105592
Javonte Williams  DAL 25.0           14.292511     14.174779
    Cam Skattebo  NYG 23.0           12.958871     15.962500
    Tyjae Spears  TEN 24.0           12.857646      8.833635
     Breece Hall  NYJ 24.0           12.649007     13.872717


In [5]:
# ── WR ────────────────────────────────────────────────────────────────────────
wr_preds, wr_cv = run_position('WR', 'wr_full.pkl', 'wr_features.pkl')


  WR
  Player-seasons: 437 | Features: 117
  Training rows (next_ppg known): 199

  Walk-forward CV:
    Predicting 2024 (n=43): Ridge MAE=3.83 | XGB MAE=3.48 | Blend MAE=3.55
    Predicting 2025 (n=25): Ridge MAE=4.03 | XGB MAE=2.66 | Blend MAE=3.23

  Selected strategy: XGB

  2026 Predictions (top 10):
        player_name team  age  predicted_ppg_2026  weighted_ppg
  Amon-Ra St. Brown  DET 25.0           18.701662     18.220239
      Ja'Marr Chase  CIN 25.0           18.190958     20.554971
       Drake London  ATL 24.0           15.862422     16.445599
     George Pickens  DAL 24.0           15.489745     15.111765
   Jameson Williams  DET 24.0           14.890789     12.888235
        Zay Flowers  BAL 24.0           14.773244     14.664706
Marvin Harrison Jr.  ARI 23.0           14.182670     10.797317
        CeeDee Lamb  DAL 26.0           13.741130     15.869255
        Rashee Rice   KC 25.0           12.793871     18.512500
        Alec Pierce  IND 25.0           12.511790   

In [6]:
# ── TE ────────────────────────────────────────────────────────────────────────
te_preds, te_cv = run_position('TE', 'te_full.pkl', 'te_features.pkl')


  TE
  Player-seasons: 241 | Features: 101
  Training rows (next_ppg known): 118

  Walk-forward CV:
    Predicting 2024 (n=30): Ridge MAE=2.60 | XGB MAE=1.80 | Blend MAE=2.13
    Predicting 2025 (n=18): Ridge MAE=2.09 | XGB MAE=2.42 | Blend MAE=2.24

  Selected strategy: BLEND

  2026 Predictions (top 10):
      player_name team  age  predicted_ppg_2026  weighted_ppg
     Brock Bowers   LV 22.0           14.474244     14.763055
       Kyle Pitts  ATL 24.0           13.456949     11.013235
     Trey McBride  ARI 25.0           12.815941     17.350846
     Tyler Warren  IND 23.0           11.990401     11.088235
Harold Fannin Jr.  CLE 21.0           11.718362     11.775000
      Sam LaPorta  DET 24.0           10.037728     11.877778
     Chig Okonkwo  TEN 26.0            9.497173      7.249613
       Cade Otton   TB 26.0            9.435533      8.146667
     Greg Dulcich  MIA 25.0            9.373851      7.571053
     Travis Kelce   KC 35.0            9.338140     11.247059


In [7]:
# ── Accuracy Summary + Combined Rankings ─────────────────────────────────────

print('\nACCURACY SUMMARY (MAE = avg points per game off)')
print('-' * 60)
for pos, cv in [('QB', qb_cv), ('RB', rb_cv), ('WR', wr_cv), ('TE', te_cv)]:
    for season, r in cv.items():
        best = min(r['ridge_mae'], r['xgb_mae'], r['blend_mae'])
        print(f'  {pos} predicting {season}: best MAE = {best:.2f} ppg (n={r["n"]})')

# Combine all predictions into one rankings table
all_preds = []
for pos, df in [('QB', qb_preds), ('RB', rb_preds), ('WR', wr_preds), ('TE', te_preds)]:
    df = df.copy()
    df['position'] = pos
    all_preds.append(df)

combined = pd.concat(all_preds, ignore_index=True)
combined = combined.sort_values('predicted_ppg_2026', ascending=False).reset_index(drop=True)
combined['rank'] = combined.index + 1

combined.to_pickle(f'{OUTPUT_DIR}/combined_predictions_ppr.pkl')

print('\nTOP 30 PPR PREDICTIONS FOR 2026')
print('-' * 60)
print(combined[['rank','player_name','position','team','age',
                'predicted_ppg_2026','weighted_ppg','strategy']]
      .head(30).to_string(index=False))

print('\nSaved:')
print('  combined_predictions_ppr.pkl')
print('  qb/rb/wr/te _predictions_ppr.pkl')
print('  qb/rb/wr/te _ridge_ppr.pkl + _xgb_ppr.pkl')


ACCURACY SUMMARY (MAE = avg points per game off)
------------------------------------------------------------
  QB predicting 2024: best MAE = 2.27 ppg (n=12)
  QB predicting 2025: best MAE = 4.21 ppg (n=5)
  RB predicting 2024: best MAE = 3.48 ppg (n=29)
  RB predicting 2025: best MAE = 3.58 ppg (n=14)
  WR predicting 2024: best MAE = 3.48 ppg (n=43)
  WR predicting 2025: best MAE = 2.66 ppg (n=25)
  TE predicting 2024: best MAE = 1.80 ppg (n=30)
  TE predicting 2025: best MAE = 2.09 ppg (n=18)

TOP 30 PPR PREDICTIONS FOR 2026
------------------------------------------------------------
 rank         player_name position team  age  predicted_ppg_2026  weighted_ppg strategy
    1   Amon-Ra St. Brown       WR  DET 25.0           18.701662     18.220239      xgb
    2       Ja'Marr Chase       WR  CIN 25.0           18.190958     20.554971      xgb
    3        Jahmyr Gibbs       RB  DET 23.0           18.027151     21.682353    blend
    4       Ashton Jeanty       RB   LV 21.0        